# Detecting Curves

In [1]:
import branchpoint as bp
import pygfx as gfx
import numpy as np
from tinygrad import Tensor, nn
import fastplotlib as fpl
import numpy as np

To silence this warning, use a fully namespaced name.
Unable to find extension: VK_EXT_physical_device_drm
Detected skylake derivative running on mesa i915. Clears to srgb textures will use manual shader clears.
Detected skylake derivative running on mesa i915. Clears to srgb textures will use manual shader clears.


## Install a shared device

In [2]:
dev = bp.gpu.install()

Detected skylake derivative running on mesa i915. Clears to srgb textures will use manual shader clears.


## Data generation

In [3]:
IMG = 32
BATCH_SIZE = 64

In [4]:
def arc(
    theta: float,
    curvature: float,
    size: int = IMG,
    thickness: float = 1.1,
    jitter=(0.0, 0.0),
) -> np.ndarray:
    """One arc image, tangent direction `theta` at the centre.

    curvature 0 gives a straight line; larger values bend tighter. The circle's
    centre is placed perpendicular to the tangent at radius 1/curvature, so the
    arc passes through the middle of the frame however much it bends.
    """
    yy, xx = np.mgrid[0:size, 0:size].astype(np.float32)
    c = (size - 1) / 2
    xx = xx - c - jitter[0]
    yy = yy - c - jitter[1]

    if curvature < 1e-3:
        d = np.abs(-np.sin(theta) * xx + np.cos(theta) * yy)
    else:
        r = 1.0 / curvature
        ox, oy = -np.sin(theta) * r, np.cos(theta) * r
        d = np.abs(np.hypot(xx - ox, yy - oy) - r)

    return np.exp(-((d / thickness) ** 2)).astype(np.float32)


def make_batch(
    bs: int, rng: np.random.Generator, max_curvature: float = 0.09
) -> tuple[Tensor, Tensor]:
    """Random arcs, labelled by which of N_ORI orientation bins they fall in.

    Orientation spans pi rather than 2*pi because an arc and its 180-degree
    rotation are the same stimulus.
    """
    labels = rng.integers(0, N_ORI, bs)
    imgs = np.empty((bs, 1, IMG, IMG), np.float32)
    for i, lab in enumerate(labels):
        theta = (lab + rng.uniform(-0.35, 0.35)) * np.pi / N_ORI
        imgs[i, 0] = arc(
            theta, rng.uniform(0.0, max_curvature), jitter=rng.uniform(-2, 2, 2)
        )
    return Tensor(imgs), Tensor(labels.astype(np.int32))

## Define a simple curve detection model

In [5]:
K = 11
N_FILTERS = 8
N_ORI = 8

In [6]:
class CurveDetector:
    def __init__(self, n_filters: int = N_FILTERS, k: int = K, n_ori: int = N_ORI):
        self.n_filters, self.k, self.n_ori = n_filters, k, n_ori
        self.conv = nn.Conv2d(1, n_filters, k, padding=k // 2)
        self.fc = nn.Linear(n_filters, n_ori)
        self.acts: dict[str, Tensor] = {}

    def parameters(self):
        return nn.state.get_parameters(self)

    def filters(self) -> Tensor:
        """(n_filters, k, k) — the learned weights, ready to display.

        These live in a buffer that keeps its identity across steps, since the
        optimizer assigns into it rather than reallocating.
        """
        return self.conv.weight.reshape(self.n_filters, self.k, self.k)

    def __call__(self, x: Tensor, filter_mask: Tensor | None = None) -> Tensor:
        a = self.acts
        a.clear()
        a["input"] = x

        pre = self.conv(x)
        a["conv_pre"] = pre

        h = pre.relu()
        if filter_mask is not None:
            # Ablation as a 0/1 multiply rather than a Python branch, so it
            # stays inside the graph and gradients to a dead filter go to zero.
            h = h * filter_mask.reshape(1, self.n_filters, 1, 1)
        a["conv_act"] = h  # (B, F, IMG, IMG)

        p = h.max(axis=(2, 3))  # (B, F) peak response per filter
        a["pooled"] = p

        logits = self.fc(p)
        a["logits"] = logits
        return logits

In [7]:
def make_step(
    model: CurveDetector,
    opt,
    rng: np.random.Generator,
    bs: int = BATCH_SIZE,
    filter_mask=None,
):
    """Returns a closure running one optimizer step and giving back the loss."""

    def step(x, y) -> Tensor:
        with Tensor.train():
            opt.zero_grad()
            mask = filter_mask() if callable(filter_mask) else filter_mask
            loss = model(x, mask).sparse_categorical_crossentropy(y).backward()
            opt.step()
        return loss

    return step

In [8]:
# define model and optimizer
rng = np.random.default_rng(0)
model = CurveDetector()
opt = nn.optim.Adam(model.parameters(), lr=3e-3)
step = make_step(model, opt, rng)

## Define visualization

In [9]:
extents = [
    (0, 0.5, 0, 1),  # inputs
    (0.5, 1, 0, 0.5),  # weights
    (0.5, 1, 0.5, 1),  # loss
]

fig = fpl.Figure(
    extents=extents, size=(1_000, 500), names=["inputs", "weights", "loss"]
)

for s in fig:
    s.axes.visible = False

### Add 8x8 grid of input images

In [10]:
scene1 = fig["inputs"].scene
SCALE, GAP = 3, 6
tile = IMG * SCALE + GAP  # 32 * 3 + 6 = 102

input_tex = []
for i in range(BATCH_SIZE):
    t = bp.gpu.TinygradTensorTexture(IMG, IMG)
    x, y = (i % 8) * tile, (7 - i // 8) * tile  # row 0 on top
    scene1.add(t.as_image(clim=(0.0, 1.0), position=(x, y, 0), scale=SCALE))
    input_tex.append(t)

fig["inputs"].camera.show_object(scene1, view_dir=(0, 0, -1), up=(0, 1, 0), scale=0.9)

### Add 2x4 grid of model weights

In [11]:
scene2 = fig["weights"].scene
SCALE, GAP = 14, 20  # 11 * 14 = 154 px tiles
tile = K * SCALE + GAP

coolwarm = gfx.cm.create_colormap(
    [
        (0.230, 0.299, 0.754),  # cool blue
        (0.865, 0.865, 0.865),  # neutral grey at the midpoint
        (0.706, 0.016, 0.150),  # warm red
    ]
)

filt_tex = []
for i in range(N_FILTERS):
    t = bp.gpu.TinygradTensorTexture(K, K)  # 11 floats/row -> padded to 64 internally
    x, y = (i % 4) * tile, (1 - i // 4) * tile
    scene2.add(
        t.as_image(clim=(-0.4, 0.4), position=(x, y, 0), scale=SCALE, cmap=coolwarm)
    )
    filt_tex.append(t)

fig["weights"].camera.show_object(scene2, view_dir=(0, 0, -1), up=(0, 1, 0), scale=0.5)

### Plotting log loss

In [12]:
MAX_PTS = 400

In [13]:
xs = np.arange(MAX_PTS).astype(np.float32)
ys = np.full((MAX_PTS,), float("nan"))


loss_graphic = fig["loss"].add_line(np.vstack([xs, ys]).T)


fig["loss"].camera.maintain_aspect = False
fig["loss"].camera.show_rect(0, MAX_PTS, -5.0, 1.0)

/home/caitlinlewis/repos/fastplotlib/fastplotlib/graphics/features/_base.py:18: UserWarning: casting float64 array to float32
  warn(f"casting {array.dtype} array to float32")


In [14]:
state = {"paused": False, "step": 0}


def animate():
    if state["paused"]:
        return
    x, y = make_batch(BATCH_SIZE, rng)
    loss = step(x, y)
    loss_graphic.data[state["step"], 1] = np.log10(loss.numpy())
    state["step"] += 1
    if state["step"] % 10 == 0:  # every 10 frames is plenty
        fig["loss"].camera.show_rect(0, max(state["step"] + 10, 100), -2.5, 1.0)
    f = model.filters().realize()  # (F, k, k), weights straight off the GPU
    for i, t in enumerate(input_tex):
        t.update(x[i, 0])
    for i, t in enumerate(filt_tex):
        t.update(f[i])


fig.add_animations(animate)
fig.show(autoscale=False)

### Pause training

In [19]:
state["paused"] = True

In [20]:
state

{'paused': True, 'step': 87}

### Change the learning rate

In [17]:
opt.lr.assign(Tensor([1e-2], device=opt.lr.device)).realize()

<Tensor <UOp WEBGPU (1,) float> on WEBGPU with grad None>

In [18]:
state["paused"] = False